# ENBIOS4TIMES: Environmental Assessment of Energy System Scenarios with ENBIOS

This notebook guides TIMES model users through the process of assessing the environmental impacts of energy system scenarios using ENBIOS2 (Environmental and Bioeconomic System Analysis).

ENBIOS2 integrates **Life Cycle Assessment (LCA)** with the **MuSIASEM** (Multi-Scale Integrated Analysis of Societal and Ecosystem Metabolism) framework to evaluate energy system pathways produced by models such as TIMES.

TIMES outputs are soft-linked to ENBIOS using [**Sparks**](https://github.com/LIVENlab/Sparks), vendored in this repo under `Sparks/`.

**Workflow:**
1. What is ENBIOS?
2. What is Sparks, and what does it need from you?
3. Preparing the input folder (the basefile)
4. Soft-linking TIMES outputs to ENBIOS with Sparks
5. Running the Environmental Assessment
6. Visualizing Results

> **Status: first draft.** This notebook has been checked against the Sparks and enbios source code in this repo, but has not yet been run end-to-end against real TIMES data - see the callouts marked with a warning symbol below before running it for real.

## 1. What is ENBIOS?

ENBIOS2 is a Python-based simulation tool for the environmental and bioeconomic assessment of energy system pathways. It is designed to work with the outputs of Energy System Optimisation Models (ESOMs) such as TIMES, Calliope, or OSeMOSYS.

ENBIOS2 does not perform LCA calculations itself - it uses [Brightway2](https://docs.brightway.dev/) as its LCA engine, connected to life cycle inventory databases such as [ecoinvent](https://ecoinvent.org/).

### Key concepts

| Concept | Description |
|---------|-------------|
| **Experiment** | The top-level object that holds the hierarchy, adapters, methods and scenarios |
| **Hierarchy (dendrogram)** | A tree structure of nodes representing the energy system |
| **Structural node (leaf)** | An energy technology linked to a Brightway/ecoinvent activity |
| **Functional node** | An aggregation node (e.g. wind energy, total electricity) |
| **Adapter** | Connects ENBIOS to an LCA database (e.g. Brightway adapter for ecoinvent) |
| **Scenario** | A set of activity outputs representing one energy system pathway from TIMES |
| **Method** | An LCIA method (e.g. ReCiPe GWP1000) used to calculate environmental impacts |

### How TIMES connects to ENBIOS

TIMES produces scenario results as **energy flows** (e.g. PJ of electricity from wind, solar, gas). Each TIMES technology is mapped to an ecoinvent activity via a **basefile**. Sparks reads the basefile plus your TIMES energy output and builds the ENBIOS hierarchy, methods and scenarios for you (Sections 2-4); ENBIOS then calculates the environmental impacts of each technology and aggregates them through the hierarchy (Sections 5-6).

## 2. What is Sparks, and what does it need from you?

[Sparks](https://github.com/LIVENlab/Sparks) soft-links energy system model outputs (TIMES, Calliope) to ENBIOS. It reads:
- a **basefile** (`basefile.xlsx`) describing your technologies, their ecoinvent mapping, the hierarchy, and the LCIA methods to use
- your **energy output data** (one or more CSV/XLSX files with the flows per technology per scenario)

...and produces a single ENBIOS-ready JSON (hierarchy + adapters/methods + scenarios) that `enbios.Experiment` can run directly.

### Before running this for the first time
- **Environment**: this repo's `.venv` needs `enbios`'s `requirements.txt` installed (including `pandera`, added for Sparks) before any of these imports will work.
- **`Sparks/const/const.py`**: on import, several Sparks modules call `bd.projects.set_current(bw_project)` at *module level*, using whatever project name is currently written in `Sparks/const/const.py` - this runs before you ever construct `SoftLink(...)`. If that file still has a stale project name (e.g. from a previous user/run) and it doesn't exist as a Brightway project, `bd.projects.set_current` will silently create an empty project with that name. Check/edit `Sparks/const/const.py` to a project that exists (or that you're fine having auto-created) before the first import each session.

### Setup and imports

Sparks isn't a packaged/installable dependency (no `setup.py`/`pyproject.toml`), so its parent folder - the repo root - needs to be added to `sys.path` before it can be imported.

In [ ]:
import sys
from pathlib import Path

import bw2data
import pandas as pd

# Make the vendored Sparks_functions package importable (repo root = two levels above this notebook)
REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from enbios import Experiment, report

# Check available Brightway projects and databases
report()

In [ ]:
# Set your Brightway project (must already exist)
PROJECT_NAME = "ecoinvent_391"   # replace with your project name

bw2data.projects.set_current(PROJECT_NAME)

## 3. Preparing the input folder (the basefile)

`SoftLink` takes a **folder path**, not individual file paths: it looks for a file named exactly `basefile.xlsx` inside that folder (required), and treats every other file in the same folder as candidate energy-output data.

### `basefile.xlsx` - required sheets

**`Processors`** - maps each TIMES technology to an ecoinvent activity:

| Column | Description |
|---|---|
| `Processor` | TIMES technology code, e.g. `EWIN1N` |
| `Region` | Region of the technology |
| `@SimulationCarrier` | TIMES energy carrier |
| `ParentProcessor` | Functional group this technology belongs to (must match a `Processor` in `Dendrogram_top`) |
| `@SimulationToEcoinventFactor` | Conversion factor from TIMES units to the ecoinvent activity's unit |
| `Ecoinvent_key_code` | ecoinvent activity code |
| `File_source` | Which energy-output file this technology's data comes from |
| `geo_loc` | ecoinvent activity location |

**`Methods`** - one row per LCIA method, as a `Formula` column holding a Python tuple literal, e.g.:
`"('ReCiPe 2016 v1.03, midpoint (H)', 'climate change', 'global warming potential (GWP1000)')"`

**`Dendrogram_top`** - the hierarchy above the technologies:

| Column | Description |
|---|---|
| `Processor` | Node name (functional group or root) |
| `ParentProcessor` | Its parent node |
| `Level` | `n`, `n-1`, `n-2`, ... - depth from the leaves |

### Energy output file(s)
CSV or Excel, with (after Sparks' internal cleaning) columns `full_name`, `energy_value`, `new_units`, `scenarios` - one row per technology, per scenario, per timestep.

## 4. Soft-linking TIMES outputs to ENBIOS with Sparks

Point `SoftLink` at the folder containing `basefile.xlsx` and your energy data, and your Brightway project.

In [ ]:
from Sparks_functions.util.base import SoftLink

# Folder containing basefile.xlsx + your TIMES/Calliope energy output file(s)
INPUT_FOLDER = r"data/times_seeds"

softlink = SoftLink(INPUT_FOLDER, PROJECT_NAME)

softlink.preprocess(national=False, specify_database=False)

Sparks will report technologies that are present in the energy data but missing from `basefile.xlsx` (or whose ecoinvent activity couldn't be found). Check them before continuing - anything listed here is silently excluded from the results:

In [ ]:
softlink.exluded_techs_and_regions

You can also check the preprocessed, unit-converted energy flows before they're turned into an ENBIOS hierarchy:

In [ ]:
softlink.preprocessed_units

Now build the ENBIOS-ready JSON (hierarchy, adapters/methods from the `Methods` sheet, and scenarios from the energy data) and save it:

In [ ]:
output_json = Path("data/enbios_input.json")
output_json.parent.mkdir(parents=True, exist_ok=True)

softlink.data_for_ENBIOS(smaller_vers=False, path_save=str(output_json))

In [ ]:
softlink.enbios2_data

If you want to check the structure of this file, you can use the following [JSON schema](https://github.com/LIVENlab/enbios/blob/main/data/schema/experiment.schema.gen.json) and a [validator](https://www.jsonschemavalidator.net/)

## 5. Running the Environmental Assessment

Now that Sparks has transformed the TIMES energy data into an ENBIOS-ready JSON, run the experiment.

In [ ]:
experiment = Experiment(str(output_json))
results = experiment.run()
print(f"Ran {len(results)} scenario(s) in {experiment.execution_time}")

Export the results to CSV and inspect them as a `DataFrame`:

In [ ]:
results_path = Path("results/ENBIOS_TIMES_results.csv")
results_path.parent.mkdir(parents=True, exist_ok=True)

experiment.results_to_csv(str(results_path))

df = pd.read_csv(results_path).fillna("")
df.head()

## 6. Visualizing Results

ENBIOS provides plotting helpers built on top of `ResultsSelector`, which wraps the experiment's results into a tidy `DataFrame` for plotting. Both take the `experiment` object directly:
- `bar_plot`: total impact per scenario, one subplot per LCIA method
- `stacked_bar_plot`: impact per scenario broken down by hierarchy level (e.g. `level=2` for technology groups)

In [ ]:
from enbios.base.result_select import ResultsSelector
from enbios.base.plot_experiment import bar_plot

rs = ResultsSelector.get_result_selector(experiment)
bar_plot(experiment)
rs.base_df

In [ ]:
from enbios.base.plot_experiment import stacked_bar_plot

stacked_bar_plot(experiment, level=2);